In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in sorted(files):
        if f.endswith(('.npy', '.csv')):
            print(os.path.join(root, f))

In [2]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

COMP = '/kaggle/input/competitions/playground-series-s6e10/'
TFM  = '/kaggle/input/datasets/cdeotte/s6e10-tfm-oof-and-test-predictions/'
MINE = '/kaggle/input/notebooks/serdarakbay/notebookbd109e0038/'

train = pd.read_csv(COMP + 'train.csv')
test = pd.read_csv(COMP + 'test.csv')
y = train['satisfaction'].astype(int).values

tfm_names = ['tabpfn35', 'kumolarge', 'kumosmall', 'tabfm', 'exaone', 'limix2',
             'causilo', 'tabicl2', 'mitra2', 'realmlp', 'xgboost']

oof, pred = {}, {}
for m in ['lgb', 'xgb', 'cb']:
    oof[m], pred[m] = np.load(MINE + f'oof_{m}.npy'), np.load(MINE + f'pred_{m}.npy')
for m in tfm_names:
    o, t = np.load(TFM + f'oof_{m}.npy'), np.load(TFM + f'test_{m}.npy')
    oof[f'ext_{m}'] = o[:, -1] if o.ndim == 2 else o
    pred[f'ext_{m}'] = t[:, -1] if t.ndim == 2 else t

for m in oof:
    print(f'{m:15s} oof {oof[m].shape}  test {pred[m].shape}  AUC = {roc_auc_score(y, oof[m]):.6f}')

lgb             oof (699635,)  test (299844,)  AUC = 0.960589
xgb             oof (699635,)  test (299844,)  AUC = 0.960616
cb              oof (699635,)  test (299844,)  AUC = 0.960350
ext_tabpfn35    oof (699635,)  test (299844,)  AUC = 0.961145
ext_kumolarge   oof (699635,)  test (299844,)  AUC = 0.960036
ext_kumosmall   oof (699635,)  test (299844,)  AUC = 0.958484
ext_tabfm       oof (699635,)  test (299844,)  AUC = 0.958871
ext_exaone      oof (699635,)  test (299844,)  AUC = 0.958557
ext_limix2      oof (699635,)  test (299844,)  AUC = 0.958474
ext_causilo     oof (699635,)  test (299844,)  AUC = 0.958275
ext_tabicl2     oof (699635,)  test (299844,)  AUC = 0.958088
ext_mitra2      oof (699635,)  test (299844,)  AUC = 0.957075
ext_realmlp     oof (699635,)  test (299844,)  AUC = 0.959311
ext_xgboost     oof (699635,)  test (299844,)  AUC = 0.959852


In [3]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def stack(models, C=0.003):
    Xo = np.column_stack([logit(oof[m]) for m in models])
    Xt = np.column_stack([logit(pred[m]) for m in models])
    nested = np.zeros(len(y))
    for tr, va in skf.split(Xo, y):
        lr = LogisticRegression(C=C, max_iter=1000)
        lr.fit(Xo[tr], y[tr])
        nested[va] = lr.predict_proba(Xo[va])[:, 1]
    lr = LogisticRegression(C=C, max_iter=1000).fit(Xo, y)
    return roc_auc_score(y, nested), lr.coef_[0], lr.predict_proba(Xt)[:, 1]

sets = {
    'lgb+xgb+tabpfn': ['lgb', 'xgb', 'ext_tabpfn35'],
    'all 14':         list(oof.keys()),
}
results = {}
for name, models in sets.items():
    auc, coef, test_pred = stack(models)
    results[name] = (auc, test_pred)
    print(f'\n{name}: nested CV = {auc:.5f}')
    for m, c in sorted(zip(models, coef), key=lambda t: -abs(t[1])):
        print(f'   {m:15s} {c:+.3f}')


lgb+xgb+tabpfn: nested CV = 0.96149
   ext_tabpfn35    +0.666
   xgb             +0.180
   lgb             +0.156

all 14: nested CV = 0.96158
   ext_tabpfn35    +0.605
   ext_tabfm       +0.255
   ext_causilo     -0.244
   xgb             +0.208
   ext_limix2      +0.203
   ext_kumosmall   -0.158
   ext_mitra2      -0.137
   lgb             +0.123
   ext_kumolarge   +0.120
   ext_exaone      -0.065
   ext_xgboost     +0.055
   ext_tabicl2     -0.042
   ext_realmlp     +0.027
   cb              +0.024


In [4]:
for name, (auc, test_pred) in results.items():
    fname = 'submission_lr_' + name.replace('+', '_').replace(' ', '') + '.csv'
    pd.DataFrame({'id': test['id'], 'satisfaction': test_pred}).to_csv(fname, index=False)
    print(fname, f'{auc:.5f}')

submission_lr_lgb_xgb_tabpfn.csv 0.96149
submission_lr_all14.csv 0.96158


In [5]:
pd.DataFrame({'id': test['id'], 'satisfaction': results['all 14'][1]}).to_csv('submission.csv', index=False)
print(pd.read_csv('submission.csv').shape)

(299844, 2)


In [6]:
LIB = '/kaggle/input/datasets/cdeotte/s6e10-124-public-oof-and-test-predictions/'
files = sorted(os.listdir(LIB))
print(len(files), 'files, e.g.', files[:3], '...', files[-3:])

lib_oof, lib_pred, skipped = {}, {}, []
for f in files:
    if not (f.startswith('oof_') and f.endswith('.npy')):
        continue
    name = f[4:-4]
    tf = f'test_{name}.npy'
    if tf not in files:
        skipped.append(name); continue
    o, t = np.load(LIB + f), np.load(LIB + tf)
    o = o[:, -1] if o.ndim == 2 else o
    t = t[:, -1] if t.ndim == 2 else t
    if len(o) == len(y) and len(t) == len(test):
        lib_oof[f'lib_{name}'], lib_pred[f'lib_{name}'] = o, t
    else:
        skipped.append(name)

lib_auc = {m: roc_auc_score(y, lib_oof[m]) for m in lib_oof}
print(f'\n{len(lib_oof)} loaded, skipped: {skipped}')
for m, a in sorted(lib_auc.items(), key=lambda t: -t[1])[:10]:
    print(f'{m:30s} {a:.5f}')
print('lowest:', f'{min(lib_auc.values()):.5f}')

248 files, e.g. ['oof_ar_cat_5f.npy', 'oof_ar_lgb_5f.npy', 'oof_ar_realmlp_5f.npy'] ... ['test_th_xgb.npy', 'test_wx_context_numeric.npy', 'test_wx_rating_segments.npy']

124 loaded, skipped: []
lib_nj_01                      0.96174
lib_sa_rmlp_v4aux_k10s2        0.96141
lib_sa_rmlp_v4aux_k10          0.96140
lib_sa_rmlp_B_k10              0.96138
lib_gp_realmlp_vb_aux          0.96134
lib_gp_realmlp_vb_aux_s2       0.96132
lib_mi_realmlp_lean            0.96131
lib_mi_xgb_d5                  0.96129
lib_sa_rmlp_B                  0.96128
lib_pfn_samanyu_route_og       0.96127
lowest: 0.94887


In [7]:
import time

pool_oof = {**oof, **lib_oof}
pool_pred = {**pred, **lib_pred}
names_all = list(pool_oof)

def feat_pair(o, t):
    if o.min() >= 0 and o.max() <= 1 and t.min() >= 0 and t.max() <= 1:
        return logit(o), logit(t)
    mu, sd = o.mean(), o.std()
    return (o - mu) / sd, (t - mu) / sd

pairs = [feat_pair(pool_oof[m], pool_pred[m]) for m in names_all]
Xo_all = np.column_stack([p[0] for p in pairs]).astype(np.float32)
Xt_all = np.column_stack([p[1] for p in pairs]).astype(np.float32)
print('pool:', Xo_all.shape, Xt_all.shape)

def nested_stack(Xo, Xt, C, topk=None):
    nested = np.zeros(len(y))
    for tr, va in skf.split(Xo, y):
        cols = np.arange(Xo.shape[1])
        if topk:
            lr = LogisticRegression(C=C, max_iter=2000).fit(Xo[tr], y[tr])
            cols = np.argsort(-np.abs(lr.coef_[0]))[:topk]
        lr = LogisticRegression(C=C, max_iter=2000).fit(Xo[tr][:, cols], y[tr])
        nested[va] = lr.predict_proba(Xo[va][:, cols])[:, 1]
    cols = np.arange(Xo.shape[1])
    if topk:
        lr = LogisticRegression(C=C, max_iter=2000).fit(Xo, y)
        cols = np.argsort(-np.abs(lr.coef_[0]))[:topk]
    lr = LogisticRegression(C=C, max_iter=2000).fit(Xo[:, cols], y)
    return roc_auc_score(y, nested), lr.predict_proba(Xt[:, cols])[:, 1]

big = {}
for C, topk in [(0.003, None), (0.01, None), (0.01, 50)]:
    t0 = time.time()
    auc, tp = nested_stack(Xo_all, Xt_all, C, topk)
    key = f'C{C}_top{topk or "all"}'
    big[key] = (auc, tp)
    print(f'{key:16s} nested CV = {auc:.5f}   ({time.time() - t0:.0f}s)')

pool: (699635, 138) (299844, 138)
C0.003_topall    nested CV = 0.96209   (78s)
C0.01_topall     nested CV = 0.96208   (81s)
C0.01_top50      nested CV = 0.96207   (119s)


In [8]:
best_auc, best_pred = big['C0.003_topall']
pd.DataFrame({'id': test['id'], 'satisfaction': best_pred}).to_csv('submission.csv', index=False)
pd.DataFrame({'id': test['id'], 'satisfaction': best_pred}).to_csv('submission_lib138.csv', index=False)
print(pd.read_csv('submission.csv').shape, f'{best_auc:.5f}')

(299844, 2) 0.96209


In [9]:
import glob

def load_pairs(root, prefix):
    oo, tt, skipped = {}, {}, []
    for dirpath, _, files in os.walk(root):
        fs = set(files)
        for f in sorted(files):
            if not (f.startswith('oof_') and f.endswith('.npy')):
                continue
            name = f[4:-4]
            tf = next((c for c in (f'test_{name}.npy', f'pred_{name}.npy') if c in fs), None)
            if tf is None:
                skipped.append(name); continue
            o, t = np.load(os.path.join(dirpath, f)), np.load(os.path.join(dirpath, tf))
            o = o[:, -1] if o.ndim == 2 else o
            t = t[:, -1] if t.ndim == 2 else t
            if len(o) == len(y) and len(t) == len(test):
                oo[prefix + name], tt[prefix + name] = o, t
            else:
                skipped.append(name)
    return oo, tt, skipped

roots = glob.glob('/kaggle/input/datasets/*/aviator*') + glob.glob('/kaggle/input/datasets/*/*route-tfm*')
print('found folders:', roots)

new_oof, new_pred = {}, {}
for r in roots:
    pre = 'av_' if 'aviator' in r else 'km_'
    o, t, sk = load_pairs(r, pre)
    new_oof.update(o); new_pred.update(t)
    print(f'{r}: {len(o)} models, skipped: {sk}')

for m in sorted(new_oof, key=lambda k: -roc_auc_score(y, new_oof[k])):
    print(f'{m:30s} AUC = {roc_auc_score(y, new_oof[m]):.5f}')

pairs_new = [feat_pair(new_oof[m], new_pred[m]) for m in new_oof]
Xo2 = np.hstack([Xo_all, np.column_stack([p[0] for p in pairs_new]).astype(np.float32)])
Xt2 = np.hstack([Xt_all, np.column_stack([p[1] for p in pairs_new]).astype(np.float32)])
print('new pool:', Xo2.shape)

t0 = time.time()
auc2, tp2 = nested_stack(Xo2, Xt2, 0.003)
big['plus_new'] = (auc2, tp2)
print(f'138 + new: nested CV = {auc2:.5f}   ({time.time() - t0:.0f}s)   base: 0.96209')

found folders: ['/kaggle/input/datasets/abdullahsafwan333/aviator-oof-dataset', '/kaggle/input/datasets/koumeimaki/s6e10-route-tfm-nn-oof']
/kaggle/input/datasets/abdullahsafwan333/aviator-oof-dataset: 19 models, skipped: []
/kaggle/input/datasets/koumeimaki/s6e10-route-tfm-nn-oof: 7 models, skipped: []
av_our_tabpfn_og               AUC = 0.96168
av_our_tabpfn_catfd10          AUC = 0.96160
av_our_xgb_v2_80c              AUC = 0.96143
av_our_xgb_v1_136c             AUC = 0.96142
av_our_xgb_og                  AUC = 0.96142
av_our_cat_v3_91c              AUC = 0.96140
av_our_xgb_v3_91c              AUC = 0.96138
av_our_tabpfn_te_cnt           AUC = 0.96138
av_our_xgb_seg                 AUC = 0.96136
av_our_zoo_lgb_80c             AUC = 0.96134
av_our_zoo_cat_80c             AUC = 0.96134
km_rmlp_f_nc                   AUC = 0.96130
km_cat_w1                      AUC = 0.96122
km_lgb_w1                      AUC = 0.96120
km_xgb_w1                      AUC = 0.96120
km_tabpfn_catfd10   

In [10]:
pd.DataFrame({'id': test['id'], 'satisfaction': tp2}).to_csv('submission.csv', index=False)
pd.DataFrame({'id': test['id'], 'satisfaction': tp2}).to_csv('submission_lib164.csv', index=False)
print(pd.read_csv('submission.csv').shape)

(299844, 2)


In [11]:
# is the lib164 vs lib138 LB difference real or noise?
from scipy.stats import spearmanr
rho = spearmanr(tp2, big['C0.003_topall'][1]).correlation
sd = 0.0012 * np.sqrt(1 - rho)
print(f'rho = {rho:.6f}   LB diff noise (sd) = {sd:.6f}   2-sd = {2*sd:.6f}')
print('CV diff = 0.00009, LB diff = 0.00001')

rho = 0.999300   LB diff noise (sd) = 0.000032   2-sd = 0.000063
CV diff = 0.00009, LB diff = 0.00001


In [12]:
# sanity check: oof vs test distributions should match
rows = []
for name, oo, tt in [('my lgb', oof['lgb'], pred['lgb']),
                     ('ext_tabpfn35', oof['ext_tabpfn35'], pred['ext_tabpfn35'])] + \
                    [(m, new_oof[m], new_pred[m]) for m in new_oof if m.startswith('av_')]:
    rows.append((name, oo.mean(), tt.mean(), oo.std(), tt.std()))
df = pd.DataFrame(rows, columns=['model', 'oof_mean', 'test_mean', 'oof_std', 'test_std'])
df['std_ratio'] = df['test_std'] / df['oof_std']
print(df.round(4).to_string(index=False))

                   model  oof_mean  test_mean  oof_std  test_std  std_ratio
                  my lgb    0.4435     0.4436   0.4321    0.4320    0.9996
            ext_tabpfn35    0.4446     0.4446   0.4336    0.4334    0.9996
       av_our_cat_v3_91c    0.4435     0.4436   0.4332    0.4330    0.9996
         av_our_lin_170c   -0.4402    -0.4407   3.0479    3.0461    0.9994
   av_our_tabpfn_catfd10    0.4451     0.4450   0.4336    0.4333    0.9994
        av_our_tabpfn_og    0.4445     0.4445   0.4336    0.4335    0.9996
       av_our_tabpfn_raw    0.4465     0.4464   0.4335    0.4334    0.9997
    av_our_tabpfn_te_cnt    0.4458     0.4458   0.4330    0.4328    0.9996
           av_our_xgb_og    0.4436     0.4437   0.4331    0.4330    0.9997
        av_our_xgb_raw21    0.4434     0.4435   0.4302    0.4301    0.9997
av_our_xgb_raw21_bin4096    0.4434     0.4435   0.4306    0.4305    0.9996
          av_our_xgb_seg    0.4436     0.4437   0.4330    0.4329    0.9997
      av_our_xgb_v1_136c